# Module 1: Telco Customer Churn Prediction & Decision Modeling

Welcome to **Module 1**! In this practical lab, you will analyze customer churn for a telecommunications provider (`Telco-Customer-Churn.csv` — 7,043 customers, 21 columns).

### Business Brief:
A telecom company loses ~25% of its customers annually. Winning new customers costs 5x more than retaining existing ones. Your brief:
> *"Predict which customers are likely to churn, identify root causes, and determine the optimal intervention cutoff for retention calls."*

---

### Assignment Tasks Overview:
1. **Task 1**: Load `Telco-Customer-Churn.csv`, inspect schema, convert `TotalCharges` strings to float, and impute missing values.
2. **Task 2**: Exploratory Data Analysis — Analyze churn rates across contract types, tenure buckets, and monthly charges.
3. **Task 3**: Preprocessing & Feature Encoding — One-hot encode categorical features and scale numeric variables.
4. **Task 4**: Machine Learning Model Training — Train Logistic Regression and Random Forest classifiers.
5. **Task 5**: Business Decision Threshold Modeling — Evaluate Precision-Recall trade-offs to maximize retention campaign ROI.

In [1]:
# Task 1: Load Telco Churn Dataset & Handle Data Types
import pandas as pd
import numpy as np

df = pd.read_csv('Telco-Customer-Churn.csv')

# TotalCharges is text; turn it into numbers (blanks become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')

# The 11 blanks are all new customers (tenure = 0) who have not paid yet, so fill with 0
print("Blank rows, tenure values:", df.loc[df['TotalCharges'].isna(), 'tenure'].unique())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# 1 = customer left, 0 = customer stayed
df['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)

print(f"Dataset Shape: {df.shape}")
print("Missing values left:", df.isna().sum().sum())
print("Churn Distribution:\n", df['Churn'].value_counts(normalize=True))
df.head()

Blank rows, tenure values: [0]
Dataset Shape: (7043, 22)
Missing values left: 0
Churn Distribution:
 No     0.73463
Yes    0.26537
Name: Churn, dtype: float64



<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>customerID</th>
      <th>gender</th>
      <th>SeniorCitizen</th>
      <th>Partner</th>
      <th>Dependents</th>
      <th>tenure</th>
      <th>PhoneService</th>
      <th>MultipleLines</th>
      <th>InternetService</th>
      <th>OnlineSecurity</th>
      <th>OnlineBackup</th>
      <th>DeviceProtection</th>
      <th>TechSupport</th>
      <th>StreamingTV</th>
      <th>StreamingMovies</th>
      <th>Contract</th>
      <th>PaperlessBilling</th>
      <th>PaymentMethod</th>
      <th>MonthlyCharges</th>
      <th>TotalCharges</th>
      <th>Churn</th>
      <th>Churn_Numeric</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
     

### Task 2: Exploratory Data Analysis & Churn Drivers
Analyze how Contract Type, Tenure, and Internet Service impact churn rates.

In [2]:
# Task 2: Churn Analysis by Contract, Tenure & Monthly Charges

# 1. By contract type
contract_churn = df.groupby('Contract')['Churn_Numeric'].agg(['count', 'mean']).rename(columns={'mean': 'Churn_Rate'})
contract_churn['Churn_Rate'] = (contract_churn['Churn_Rate'] * 100).round(2).astype(str) + '%'
print("--- Churn Rate by Contract Type ---")
print(contract_churn)

# 2. By tenure (start at -1 so customers with tenure 0 are included)
df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72],
                           labels=['0-1 yr', '1-2 yrs', '2-4 yrs', '4-6 yrs'])
tenure_churn = df.groupby('TenureGroup', observed=False)['Churn_Numeric'].agg(['count', 'mean'])
tenure_churn['mean'] = (tenure_churn['mean'] * 100).round(2).astype(str) + '%'
print("\n--- Churn Rate by Tenure Group ---")
print(tenure_churn.rename(columns={'mean': 'Churn_Rate'}))

# 3. By monthly charges (split into 4 equal-sized groups)
df['ChargeGroup'] = pd.qcut(df['MonthlyCharges'], 4,
                            labels=['Low', 'Medium', 'High', 'Very High'])
charge_churn = df.groupby('ChargeGroup', observed=False)['Churn_Numeric'].agg(['count', 'mean'])
charge_churn['mean'] = (charge_churn['mean'] * 100).round(2).astype(str) + '%'
print("\n--- Churn Rate by Monthly Charges ---")
print(charge_churn.rename(columns={'mean': 'Churn_Rate'}))

--- Churn Rate by Contract Type ---
                count Churn_Rate
Contract                        
Month-to-month   3875     42.71%
One year         1473     11.27%
Two year         1695      2.83%

--- Churn Rate by Tenure Group ---
             count Churn_Rate
TenureGroup                  
0-1 yr        2186     47.44%
1-2 yrs       1024     28.71%
2-4 yrs       1594     20.39%
4-6 yrs       2239      9.51%

--- Churn Rate by Monthly Charges ---
             count Churn_Rate
ChargeGroup                  
Low           1762     11.24%
Medium        1766     24.58%
High          1757     37.51%
Very High     1758     32.88%



### Task 3: Model Training & Evaluation
Train a Random Forest classifier and generate a Confusion Matrix.

In [4]:
# Task 3: Preprocessing & Baseline Classifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Number columns and text columns (leave out customerID and the answer columns)
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

# Turn each text column into yes/no (1/0) columns
X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=int)
y = df['Churn_Numeric']
print("Feature table shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Feature table shape: (7043, 30)
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.62      0.50      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.78      1409

Confusion Matrix:
 [[920 115]
 [188 186]]



### **Task 4**: Machine Learning Model Training — Train Logistic Regression and Random Forest classifiers.

In [5]:
# Task 4: Train Logistic Regression and Random Forest
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Logistic Regression needs scaling. Putting the scaler inside a Pipeline means
# it learns the sizes from the training data only, so the test data stays unseen.
log_reg = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])
log_reg.fit(X_train, y_train)

# Random Forest does not need scaling.
# min_samples_leaf=5 stops it from memorising tiny groups of customers.
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

for name, m in [('Logistic Regression', log_reg), ('Random Forest', rf)]:
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1]
    print(f"\n===== {name} =====")
    print("Ranking score (AUC):", round(roc_auc_score(y_test, prob), 3))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, digits=3))

# What pushes customers to leave? (Logistic Regression weights)
weights = pd.Series(log_reg.named_steps['model'].coef_[0], index=X.columns).sort_values()
print("\nTop reasons to STAY:\n", weights.head(5).round(2))
print("\nTop reasons to LEAVE:\n", weights.tail(5).round(2))

# What does Random Forest lean on most?
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nRandom Forest top 10:\n", importance.head(10).round(3))


===== Logistic Regression =====
Ranking score (AUC): 0.842
[[925 110]
 [163 211]]
              precision    recall  f1-score   support

           0      0.850     0.894     0.871      1035
           1      0.657     0.564     0.607       374

    accuracy                          0.806      1409
   macro avg      0.754     0.729     0.739      1409
weighted avg      0.799     0.806     0.801      1409


===== Random Forest =====
Ranking score (AUC): 0.843
[[939  96]
 [177 197]]
              precision    recall  f1-score   support

           0      0.841     0.907     0.873      1035
           1      0.672     0.527     0.591       374

    accuracy                          0.806      1409
   macro avg      0.757     0.717     0.732      1409
weighted avg      0.797     0.806     0.798      1409


Top reasons to STAY:
 tenure               -1.25
MonthlyCharges       -0.94
Contract_Two year    -0.58
Contract_One year    -0.28
OnlineSecurity_Yes   -0.12
dtype: float64

Top reasons 

## **Task 5**: Business Decision Threshold Modeling — Evaluate Precision-Recall trade-offs to maximize retention campaign ROI.

In [6]:
# Task 5: Business Decision Threshold Modeling
from sklearn.model_selection import cross_val_predict

# ---- Example business numbers (change these if your module gives real ones) ----
CALL_COST     = 50     # cost of one retention call + offer, in $
SUCCESS_RATE  = 0.30   # share of real leavers who stay after the call
MONTHS_VALUE  = 12     # a kept customer is worth this many months of their bill

def money_table(prob, y_true, monthly_bill):
    rows = []
    for cut in np.arange(0.05, 0.96, 0.05):
        called   = prob >= cut
        caught   = called & (y_true == 1)     # called, and really leaving
        wasted   = called & (y_true == 0)     # called, but was staying anyway
        saved    = (monthly_bill[caught] * MONTHS_VALUE * SUCCESS_RATE).sum()
        cost     = called.sum() * CALL_COST
        rows.append({
            'cutoff': round(cut, 2),
            'called': called.sum(),
            'caught': caught.sum(),
            'wasted_calls': wasted.sum(),
            'precision': round(caught.sum() / max(called.sum(), 1), 3),
            'recall': round(caught.sum() / y_true.sum(), 3),
            'money_saved': round(saved),
            'call_cost': cost,
            'net_gain': round(saved - cost),
        })
    return pd.DataFrame(rows)

# Step 1: choose the cutoff using the TRAINING data only.
# cross_val_predict gives each training customer a score from a model that did not see them.
train_prob = cross_val_predict(log_reg, X_train, y_train, cv=5, method='predict_proba')[:, 1]
train_table = money_table(train_prob, y_train.values, X_train['MonthlyCharges'].values)
best_cut = train_table.loc[train_table['net_gain'].idxmax(), 'cutoff']
print(f"Best cutoff (chosen on training data): {best_cut}")

# Step 2: check that cutoff on the TEST data, which has not been used for any choice.
test_prob = log_reg.predict_proba(X_test)[:, 1]
test_table = money_table(test_prob, y_test.values, X_test['MonthlyCharges'].values)
print(test_table.to_string(index=False))

Best cutoff (chosen on training data): 0.25
 cutoff  called  caught  wasted_calls  precision  recall  money_saved  call_cost  net_gain
   0.05    1024     369           655      0.360   0.987        96891      51200     45691
   0.10     871     354           517      0.406   0.947        92506      43550     48956
   0.15     779     343           436      0.440   0.917        89762      38950     50812
   0.20     685     320           365      0.467   0.856        85683      34250     51433
   0.25     608     303           305      0.498   0.810        82232      30400     51832
   0.30     543     282           261      0.519   0.754        77952      27150     50802
   0.35     487     265           222      0.544   0.709        73268      24350     48918
   0.40     441     250           191      0.567   0.668        69504      22050     47454
   0.45     384     229           155      0.596   0.612        64462      19200     45262
   0.50     321     211           110      0.6

In [11]:
# ─── Custom Dataset Loaded: Telco-Customer-Churn.csv ───
import pandas as pd

df_custom = pd.read_csv('Telco-Customer-Churn.csv')
print(f"Dataset Shape: {df_custom.shape}")
df_custom.head()